# Module 4 Assignment: Database Integration Using Jupyter Notebook
**Course:** Database Integration  
**Database:** calval (PostgreSQL)  
**Purpose:** Demonstrate CRUD operations and ETL concepts using Python and PostgreSQL

---

## 1. Connect to PostgreSQL Database
We use the `psycopg2` library to establish a connection to the local PostgreSQL `calval` database.

In [ ]:
# Install psycopg2 if not already installed
# !pip install psycopg2-binary

import psycopg2
from psycopg2 import sql, OperationalError
import pandas as pd

# --- Connection Configuration ---
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "database": "calval",
    "user":     "postgres",      # update if your username differs
    "password": "YOUR_PASSWORD"       # update to your actual password
}

def get_connection():
    """Return a new psycopg2 connection using DB_CONFIG."""
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        print("✅ Connection successful!")
        return conn
    except OperationalError as e:
        print(f"❌ Connection failed: {e}")
        return None

# Test the connection
conn = get_connection()
if conn:
    conn.close()

---
## 2. CREATE – Insert a New Record
The `create_nutrient_record()` function inserts a new row into the **Nutrient** table.

In [ ]:
def create_nutrient_record(nutrient_id, nutrient_name, unit_name, nutrient_nbr, rank):
    """
    CREATE – Insert a new record into the Nutrient table.

    Parameters
    ----------
    nutrient_id   : int   – Primary key
    nutrient_name : str   – Full nutrient name
    unit_name     : str   – Unit of measure (e.g. 'g', 'mg')
    nutrient_nbr  : float – USDA nutrient number
    rank          : int   – Display rank
    """
    insert_query = """
        INSERT INTO nutrient (id, name, unit_name, nutrient_nbr, rank)
        VALUES (%s, %s, %s, %s, %s)
    """
    conn = get_connection()
    if conn is None:
        return

    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute(insert_query,
                            (nutrient_id, nutrient_name, unit_name, nutrient_nbr, rank))
        print(f"✅ Record inserted – id={nutrient_id}, name='{nutrient_name}'")
    except Exception as e:
        print(f"❌ Insert failed: {e}")
    finally:
        conn.close()


# --- Execute CREATE ---
create_nutrient_record(
    nutrient_id   = 99901,
    nutrient_name = "Test Nutrient",
    unit_name     = "mg",
    nutrient_nbr  = 999.01,
    rank          = 9990
)

---
## 3. READ – Retrieve a Record
The `read_nutrient_record()` function queries the **Nutrient** table by primary key and returns the row as a Pandas DataFrame.

In [ ]:
def read_nutrient_record(nutrient_id):
    """
    READ – Fetch a single record from the Nutrient table by id.

    Parameters
    ----------
    nutrient_id : int – Primary key to look up

    Returns
    -------
    pd.DataFrame with the matching row, or an empty DataFrame if not found.
    """
    select_query = """
        SELECT id, name, unit_name, nutrient_nbr, rank
        FROM   nutrient
        WHERE  id = %s
    """
    conn = get_connection()
    if conn is None:
        return pd.DataFrame()

    try:
        with conn.cursor() as cur:
            cur.execute(select_query, (nutrient_id,))
            row = cur.fetchone()
            if row:
                cols = [desc[0] for desc in cur.description]
                df = pd.DataFrame([row], columns=cols)
                print(f"✅ Record found for id={nutrient_id}")
                return df
            else:
                print(f"⚠️  No record found for id={nutrient_id}")
                return pd.DataFrame()
    except Exception as e:
        print(f"❌ Read failed: {e}")
        return pd.DataFrame()
    finally:
        conn.close()


# --- Execute READ ---
result = read_nutrient_record(99901)
display(result)

---
## 4. UPDATE – Modify an Existing Record
The `update_nutrient_record()` function changes the `name` and `unit_name` of an existing row.

In [ ]:
def update_nutrient_record(nutrient_id, new_name, new_unit_name):
    """
    UPDATE – Modify name and unit_name for a given nutrient id.

    Parameters
    ----------
    nutrient_id   : int – Primary key of the row to update
    new_name      : str – Replacement nutrient name
    new_unit_name : str – Replacement unit of measure
    """
    update_query = """
        UPDATE nutrient
        SET    name      = %s,
               unit_name = %s
        WHERE  id        = %s
    """
    conn = get_connection()
    if conn is None:
        return

    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute(update_query, (new_name, new_unit_name, nutrient_id))
                if cur.rowcount == 0:
                    print(f"⚠️  No row updated – id={nutrient_id} not found.")
                else:
                    print(f"✅ Record updated – id={nutrient_id}, "
                          f"new name='{new_name}', new unit='{new_unit_name}'")
    except Exception as e:
        print(f"❌ Update failed: {e}")
    finally:
        conn.close()


# --- Execute UPDATE ---
update_nutrient_record(
    nutrient_id   = 99901,
    new_name      = "Test Nutrient (Updated)",
    new_unit_name = "g"
)

# Verify the change
result = read_nutrient_record(99901)
display(result)

---
## 5. DELETE – Remove a Record
The `delete_nutrient_record()` function removes a row from the **Nutrient** table by primary key.

In [ ]:
def delete_nutrient_record(nutrient_id):
    """
    DELETE – Remove a record from the Nutrient table by id.

    Parameters
    ----------
    nutrient_id : int – Primary key of the row to delete
    """
    delete_query = """
        DELETE FROM nutrient
        WHERE  id = %s
    """
    conn = get_connection()
    if conn is None:
        return

    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute(delete_query, (nutrient_id,))
                if cur.rowcount == 0:
                    print(f"⚠️  No row deleted – id={nutrient_id} not found.")
                else:
                    print(f"✅ Record deleted – id={nutrient_id}")
    except Exception as e:
        print(f"❌ Delete failed: {e}")
    finally:
        conn.close()


# --- Execute DELETE ---
delete_nutrient_record(99901)

# Confirm deletion
result = read_nutrient_record(99901)
print("Record exists after delete:", not result.empty)

---
## 6. Full Integration Verification
Run all four CRUD operations in sequence to confirm end-to-end integration.

In [ ]:
print("=" * 55)
print("  FULL CRUD INTEGRATION TEST")
print("=" * 55)

TEST_ID = 99902

# ── CREATE ──────────────────────────────────────────────
print("\n[CREATE]")
create_nutrient_record(
    nutrient_id   = TEST_ID,
    nutrient_name = "Integration Test Nutrient",
    unit_name     = "mcg",
    nutrient_nbr  = 999.02,
    rank          = 9991
)

# ── READ ────────────────────────────────────────────────
print("\n[READ – after CREATE]")
display(read_nutrient_record(TEST_ID))

# ── UPDATE ──────────────────────────────────────────────
print("\n[UPDATE]")
update_nutrient_record(
    nutrient_id   = TEST_ID,
    new_name      = "Integration Test Nutrient (Updated)",
    new_unit_name = "IU"
)

# ── READ ────────────────────────────────────────────────
print("\n[READ – after UPDATE]")
display(read_nutrient_record(TEST_ID))

# ── DELETE ──────────────────────────────────────────────
print("\n[DELETE]")
delete_nutrient_record(TEST_ID)

# ── FINAL VERIFICATION ──────────────────────────────────
print("\n[READ – after DELETE]")
final = read_nutrient_record(TEST_ID)
print("Record still exists:", not final.empty)

print("\n" + "=" * 55)
print("  ALL CRUD OPERATIONS VERIFIED ✅")
print("=" * 55)

---
## 7. ETL Analysis: How CRUD Functions Support Extract, Transform, Load

### Overview
ETL (Extract, Transform, Load) is a data-integration pattern used to move and reshape data between systems. The four CRUD functions built above map directly onto each ETL phase.

---

### Extract
The **READ** function (`read_nutrient_record`) is the extraction engine. In a real ETL pipeline it would be generalized to pull entire datasets rather than single rows:

```python
# Extract: pull all nutrients from the source calval database
def extract_all_nutrients():
    conn = get_connection()
    df = pd.read_sql("SELECT * FROM nutrient", conn)
    conn.close()
    return df
```

This approach pulls raw nutrient data from the USDA/calval tables into Pandas DataFrames, making the data available for transformation before any destination write.

---

### Transform
Between extraction and loading, Python/Pandas applies business rules to the DataFrame — no direct database call needed:

```python
def transform_nutrients(df):
    df['name'] = df['name'].str.strip().str.title()   # normalize casing
    df = df.dropna(subset=['nutrient_nbr'])            # drop incomplete rows
    df = df[df['rank'] > 0]                            # filter invalid ranks
    df['nutrient_nbr'] = df['nutrient_nbr'].round(3)   # standardize precision
    return df
```

Typical transformations for the Nutrient dataset include: standardizing unit names (`'MG'` → `'mg'`), filtering out placeholder test rows, enriching rows with a `last_updated` timestamp, and joining with food-portion data for downstream analytics.

---

### Load
The **CREATE** and **UPDATE** functions handle loading. A real pipeline uses **upsert** logic — insert if new, update if the key already exists:

```python
def load_nutrient(row):
    existing = read_nutrient_record(row['id'])
    if existing.empty:
        create_nutrient_record(row['id'], row['name'],
                               row['unit_name'], row['nutrient_nbr'], row['rank'])
    else:
        update_nutrient_record(row['id'], row['name'], row['unit_name'])
```

---

### Delete / Archive in ETL
The **DELETE** function supports *soft-purge* and *delta-load* strategies: records that no longer appear in the source feed can be removed or flagged as inactive in the destination, keeping the calval database synchronized with upstream USDA nutrient releases.

---

### Full ETL Pipeline Sketch

```python
def run_etl_pipeline(source_df):
    """End-to-end ETL for the Nutrient table."""
    # 1. Extract – source_df already loaded from CSV or remote DB
    print(f"Extracted {len(source_df)} rows from source.")

    # 2. Transform
    clean_df = transform_nutrients(source_df)
    print(f"Transformed to {len(clean_df)} clean rows.")

    # 3. Load
    loaded, skipped = 0, 0
    for _, row in clean_df.iterrows():
        try:
            load_nutrient(row)
            loaded += 1
        except Exception as e:
            print(f"  Skipped id={row['id']}: {e}")
            skipped += 1

    print(f"ETL complete – loaded: {loaded}, skipped: {skipped}")
```

This pattern is directly applicable to loading the USDA FoodData Central CSV exports into the calval schema, keeping nutrient reference data current without manual DBeaver imports.

---

## 8. Backup
Run the cell below inside Codio to create a `.sql` dump of the `calval` database for submission to GitHub.

In [ ]:
import subprocess, datetime, os

timestamp  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
backup_file = f"calval_backup_{timestamp}.sql"

result = subprocess.run(
    ["pg_dump", "-U", "postgres", "-d", "calval", "-f", backup_file],
    capture_output=True, text=True
)

if result.returncode == 0:
    size = os.path.getsize(backup_file)
    print(f"✅ Backup created: {backup_file}  ({size:,} bytes)")
    print(f"   Upload '{backup_file}' to your GitHub 'module4-1-LastName' repository.")
else:
    print(f"❌ pg_dump error: {result.stderr}")